In [ ]:
from pathlib import Path
import geopandas as gpd
import rasterio
import numpy as np
from shapely.geometry import box
from rasterio.mask import mask
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, NullLocator
import sys, pathlib
sys.path.append(str(pathlib.Path("../../../robyns_libraries").resolve()))
import Robyn_paper_2_defs
from matplotlib import patheffects as pe, font_manager as fm

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")

In [ ]:
jamaica_boundary_path = base_path / "dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(jamaica_boundary.crs)

output_dir = base_path / "dphil_paper_2/results/flood_damage_results/expected_annual_damages_catchment"
out_dir = base_path / "dphil_paper_2/results/figures"

In [ ]:
damage_reduction_min = base_path / "dphil_paper_2/processed_data/nbs_river_catchment/damage_reduction/damage_reduction_min.tif"
damage_reduction_max = base_path /  "dphil_paper_2/processed_data/nbs_river_catchment/damage_reduction/damage_reduction_max.tif"

In [ ]:
catchments = gpd.read_file(base_path / "dphil_paper_2/processed_data/major_river_catchments/major_basins_plus_coastal_unionized_final.gpkg")
# catchments = gpd.read_file(catchments_unionized_final)
catchments.head()
print("Catchments:", len(catchments))


In [ ]:
# Sum all valid pixels in the raster (J$)
with rasterio.open(damage_reduction_min) as src:
    band = src.read(1, masked=True).astype("float64")
    # apply scale/offset if present
    scale  = (src.scales[0]  if getattr(src, "scales", None)  else 1.0) or 1.0
    offset = (src.offsets[0] if getattr(src, "offsets", None) else 0.0) or 0.0
    band = band * scale + offset
    band = np.ma.masked_invalid(band)

    avoided_ead_max_total_J = float(band.sum())

avoided_ead_max_total_USD = avoided_ead_max_total_J / 150

print(f"Total avoided_ead_max (J$): {avoided_ead_max_total_J:,.0f}")
print(f"Total avoided_ead_max (USD @150): ${avoided_ead_max_total_USD:,.0f}")

In [ ]:
# 1) Zonal sum helper (sums raster values within each polygon; no area calc)
def zonal_sum_only(raster_path, gdf, id_col="catchment_uid", all_touched=False):
    rows = []
    with rasterio.open(raster_path) as src:
        gdf_proj = gdf.to_crs(src.crs).copy()
        gdf_proj["geometry"] = gdf_proj.geometry.buffer(0)  # repair invalid geoms
        rb = box(*src.bounds)
        gdf_proj = gdf_proj[gdf_proj.intersects(rb)].copy()

        nd = src.nodata
        scale = (src.scales[0] if getattr(src, "scales", None) else 1.0) or 1.0
        offset = (src.offsets[0] if getattr(src, "offsets", None) else 0.0) or 0.0

        for _, r in gdf_proj.iterrows():
            try:
                data, _ = mask(src, [r.geometry.__geo_interface__],
                               crop=True, filled=False, all_touched=all_touched)
            except ValueError:
                rows.append({id_col: r[id_col], "sum": 0.0})
                continue

            band = data[0].astype("float64")
            band = band * scale + offset  # apply scale/offset if present

            ma = np.ma.array(band, mask=np.ma.getmaskarray(band))  # keep outside masked
            if nd is not None:
                ma = np.ma.masked_where(band == nd, ma)
            ma = np.ma.masked_invalid(ma)

            rows.append({id_col: r[id_col], "sum": float(ma.sum()) if ma.count() else 0.0})

    out = pd.DataFrame(rows)
    # ensure every ID appears (zeros for non-overlapping polygons)
    all_ids = gdf[[id_col]].copy()
    out = all_ids.merge(out, on=id_col, how="left").fillna({"sum": 0.0})
    return out


In [ ]:
# 2) Compute avoided_ead sums (min/max) by numeric catchment_uid
zs_min = zonal_sum_only(damage_reduction_min, catchments, id_col="catchment_uid") \
           .rename(columns={"sum": "avoided_ead_min"})
zs_max = zonal_sum_only(damage_reduction_max, catchments, id_col="catchment_uid") \
           .rename(columns={"sum": "avoided_ead_max"})

result = (catchments.merge(zs_min, on="catchment_uid")
                   .merge(zs_max, on="catchment_uid"))

# Optional central estimate
result["avoided_ead_mid"] = result[["avoided_ead_min","avoided_ead_max"]].mean(axis=1, skipna=True)





In [ ]:
zs_min.head()

In [ ]:
result["avoided_ead_min_usd"] = result["avoided_ead_min"] / 150
result[["catchment_uid","avoided_ead_min","avoided_ead_min_usd"]].head()

out_csv = base_path / "dphil_paper_2/results/flood_damage_results/expected_annual_damages_catchment/catchment_avoided_ead_min_usd.csv"
# out_csv.parent.mkdir(parents=True, exist_ok=True)

catchment_avoided_damages = result[["catchment_uid", "avoided_ead_min", "avoided_ead_min_usd"]].copy()
catchment_avoided_damages.to_csv(out_csv, index=False)
print("Saved:", out_csv)

In [ ]:
total_avoided_ead_min_J   = float(result["avoided_ead_min"].sum(skipna=True))
total_avoided_ead_min_USD = float(result["avoided_ead_min_usd"].sum(skipna=True))

print(f"Total avoided_ead_min (J$): {total_avoided_ead_min_J:,.0f}")
print(f"Total avoided_ead_min (USD @150): ${total_avoided_ead_min_USD:,.0f}")
print(f"In USD millions: {total_avoided_ead_min_USD/1e6:,.1f} mn")

In [ ]:
# === Build one table with J$ and USD (millions) side-by-side =================

FX_JMD_PER_USD = globals().get("FX_JMD_PER_USD", 150.0)  # keep your earlier setting

# Ensure area_km2 exists (use metres CRS if available)
if "area_km2" not in result.columns:
    if (getattr(result, "crs", None) is None) or (not result.crs.is_projected):
        _r = result.to_crs(3448)  # Jamaica CRS in metres
        result["area_km2"] = _r.geometry.area / 1e6
    else:
        result["area_km2"] = result.geometry.area / 1e6

# J$ → USD (millions), append as new columns
cols_ead = ["avoided_ead_min", "avoided_ead_max", "avoided_ead_mid"]
result = result.copy()
for c in cols_ead:
    result[f"{c}_usd_mn"] = result[c] / FX_JMD_PER_USD / 1e6

# Assemble table, sort by J$ max (keep geometry out)
first = ["catchment_uid", "area_km2"] + cols_ead + [f"{c}_usd_mn" for c in cols_ead]
tbl = (
    result.drop(columns="geometry")
          .reindex(columns=[c for c in first if c in result.columns] +
                           [c for c in result.columns if c not in first])
          .sort_values("avoided_ead_max", ascending=False)
)

# OPTIONAL: save a clean (machine-friendly) CSV before renaming
clean_csv = output_dir / "catchment_avoided_ead_JMD_and_USDmn_CLEAN_min.csv"
tbl.to_csv(clean_csv, index=False)

# --- PERMANENT RENAME (adds units in headers) --------------------------------
tbl.columns = tbl.columns.str.strip()  # <-- place this immediately before rename
rename_perm = {
    "area_km2": "area (km²)",
    "avoided_ead_min": "avoided_ead_min (J$)",
    "avoided_ead_max": "avoided_ead_max (J$)",
    "avoided_ead_mid": "avoided_ead_mid (J$)",
    "avoided_ead_min_usd_mn": "avoided_ead_min (US$ mn)",
    "avoided_ead_max_usd_mn": "avoided_ead_max (US$ mn)",
    "avoided_ead_mid_usd_mn": "avoided_ead_mid (US$ mn)",
}
tbl = tbl.rename(columns=rename_perm, errors="raise")

# Pretty print (formatters must match the NEW names)
fmt = {
    "area (km²)": lambda x: f"{x:,.2f}",
    "avoided_ead_min (J$)": lambda x: f"{x:,.0f}",
    "avoided_ead_max (J$)": lambda x: f"{x:,.0f}",
    "avoided_ead_mid (J$)": lambda x: f"{x:,.0f}",
    "avoided_ead_min (US$ mn)": lambda x: f"{x:,.1f}",
    "avoided_ead_max (US$ mn)": lambda x: f"{x:,.1f}",
    "avoided_ead_mid (US$ mn)": lambda x: f"{x:,.1f}",
}
print(tbl.head(30).to_string(index=False, formatters=fmt))

# Save the DISPLAY CSV (with unit-labelled headers)
display_csv = output_dir / "catchment_avoided_ead_JMD_and_USDmn_DISPLAY_min.csv"
tbl.to_csv(display_csv, index=False)

print("Saved:", clean_csv)
print("Saved:", display_csv)

In [ ]:
# === Fig. 3(c) — Avoided EAD (US$ millions) with neutral-grey edges (Opt B) ==

mm = globals().get("mm", 1/25.4)

# Ensure USD (millions) column exists
FX_JMD_PER_USD = globals().get("FX_JMD_PER_USD", 150.0)
if "avoided_ead_min_usd_mn" not in result.columns:
    result["avoided_ead_min_usd_mn"] = result["avoided_ead_min"] / FX_JMD_PER_USD / 1e6

# Values + log scale bounds (USD millions)
vals_mn = result["avoided_ead_min_usd_mn"].astype(float)
arr = vals_mn.to_numpy(); pos = arr[arr > 0]
FLOOR_MN = 0.1
hi = np.percentile(pos, 98) if pos.size else (np.nanmax(arr) if np.isfinite(arr).any() else FLOOR_MN*10)
hi = max(hi, FLOOR_MN*2)
lo = max((np.percentile(pos, 1) if pos.size else FLOOR_MN), FLOOR_MN)
if hi <= lo: lo = hi/5
plot_vals = vals_mn.where(vals_mn > 0)

# Colormap + log scale
cmap = mpl.colormaps.get_cmap("Blues").copy()
cmap.set_bad("whitesmoke")
norm = mpl.colors.LogNorm(vmin=lo, vmax=hi)

# Figure (match land-use map size)
fig, ax = plt.subplots(figsize=(180*mm, 150*mm))
ax.set_axis_off()

# --- Base fill WITH a neutral-grey edge (shows on light polygons) ------------
result.plot(
    ax=ax, column=plot_vals, cmap=cmap, norm=norm,
    edgecolor="#BDBDBD", linewidth=0.35, zorder=1
)

# --- Island outline casing (white then dark hairline) ------------------------
try:
    outline_geom = jamaica_boundary.union_all()
except AttributeError:
    outline_geom = jamaica_boundary.unary_union
gpd.GeoSeries([outline_geom], crs=result.crs).plot(
    ax=ax, facecolor="none", edgecolor="white",   linewidth=1.2, zorder=98
)
gpd.GeoSeries([outline_geom], crs=result.crs).plot(
    ax=ax, facecolor="none", edgecolor="#222222", linewidth=0.6, zorder=99
)

# --- Colorbar ----------------------------------------------------------------
sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap); sm._A = []
cbar = fig.colorbar(sm, ax=ax, fraction=0.030, pad=0.012)
nice = np.array([0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 50, 100])
ticks = nice[(nice >= lo) & (nice <= hi)]
if ticks.size: cbar.set_ticks(ticks)
cbar.formatter = FuncFormatter(lambda x, pos: f"{x:g}")
cbar.update_ticks()
cbar.minorticks_off()
cbar.ax.yaxis.set_minor_locator(NullLocator())
cbar.ax.tick_params(which="both", width=0.35, length=2.0, labelsize=6)
cbar.set_label("Avoided EAD (US$ millions)", fontsize=6.5, labelpad=4)

# --- Title (axes title, like land-use map) -----------------------------------
# ax.set_title(
#     "Fig. 3(c) Priority catchments for forest restoration\n"
#     "based on maximum avoided EADs (US$ millions)",
#     fontsize=7, fontweight="bold", pad=6
# )

# --- Scale bar + North arrow --------------------------------------------------
Robyn_paper_2_defs.draw_scale_bar(ax, location=(0.88, 0.78), length_km=20, linewidth=0.6,
              label_offset=0.02, km_offset=0.01)
Robyn_paper_2_defs.draw_north_arrow(ax, location=(0.88, 0.86), size=0.05, fontsize=8, label_offset=0.02)

# --- Labels (Arial ≥5 pt, crisp white halo) ----------------------------------
NUM_FS, HALO_W = 5.0, 0.75
NUM_FP = fm.FontProperties(family="Arial", size=NUM_FS)
pts = result.geometry.representative_point()
try:
    mask = result.to_crs(3448).geometry.area >= 2e6  # label ≥ ~2 km²
except Exception:
    mask = ~pts.is_empty

texts = []
for (x, y, uid, show) in zip(pts.x, pts.y, result["catchment_uid"].astype(str), mask):
    if not show: 
        continue
    t = ax.text(
        x, y, uid, fontproperties=NUM_FP, ha="center", va="center", color="black",
        zorder=20, snap=True,
        path_effects=[pe.withStroke(linewidth=HALO_W, foreground="white",
                                    joinstyle="round", capstyle="round")]
    )
    texts.append(t)

try:
    from adjustText import adjust_text
    adjust_text(texts, ax=ax, only_move={'texts':'y'},
                expand_text=(1.08, 1.15), force_text=(0.06, 0.16))
except Exception:
    pass

plt.tight_layout()

# Save
fname = out_dir / "Fig3c_avoided_EAD_min_USD_millions_internal_borders_grey"
plt.savefig(fname.with_suffix(".png"), dpi=600, bbox_inches="tight", facecolor="white")
plt.savefig(fname.with_suffix(".pdf"),              bbox_inches="tight")
plt.show()
print("Saved:", fname.with_suffix(".png"))

print("Figure DPI:", fig.dpi)
print("Unique label point sizes:", sorted({t.get_fontsize() for t in texts}))
assert all(abs(t.get_fontsize() - 5.0) < 1e-9 for t in texts), "A label isn't 5 pt!"

In [ ]:
# === Fig. 3(c) — Avoided EAD (US$ millions) — LINEAR SCALE ==================


mm = globals().get("mm", 1/25.4)
FX_JMD_PER_USD = globals().get("FX_JMD_PER_USD", 150.0)

# --- USD (millions) series (robust to input variant) -------------------------
if "avoided_ead_min_usd_mn" in result.columns:
    vals_mn = result["avoided_ead_min_usd_mn"].astype(float)
elif "avoided_ead_min_usd" in result.columns:
    vals_mn = result["avoided_ead_min_usd"].astype(float) / 1e6
else:
    vals_mn = result["avoided_ead_min"].astype(float) / FX_JMD_PER_USD / 1e6

arr = vals_mn.to_numpy()
pos = arr[arr > 0]
# cap vmax at a high percentile so extreme outliers don't wash out the rest
if pos.size:
    vmax = np.percentile(pos, 98)
else:
    vmax = np.nanmax(arr) if np.isfinite(arr).any() else 1.0
vmax = max(vmax, 0.1)   # make sure it's > 0

# Mask zeros so true-zero polygons show as 'whitesmoke' (like before).
# If you prefer to *include* zeros in the lightest blue, use: vals_mn.clip(lower=0)
plot_vals = vals_mn.where(vals_mn > 0)

# --- Colormap + **linear** norm ----------------------------------------------
cmap = mpl.colormaps.get_cmap("Blues").copy()
cmap.set_bad("whitesmoke")
norm = mpl.colors.Normalize(vmin=0.0, vmax=vmax)

# --- Figure -------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(180*mm, 150*mm))
ax.set_axis_off()

# Internal borders in neutral grey so they show on light fills
result.plot(
    ax=ax, column=plot_vals, cmap=cmap, norm=norm,
    edgecolor="#BDBDBD", linewidth=0.35, zorder=1
)

# Island outline casing (safe if boundary missing)
try:
    try:
        outline_geom = jamaica_boundary.union_all()
    except AttributeError:
        outline_geom = jamaica_boundary.unary_union
    gpd.GeoSeries([outline_geom], crs=result.crs).plot(
        ax=ax, facecolor="none", edgecolor="white",   linewidth=1.2, zorder=98
    )
    gpd.GeoSeries([outline_geom], crs=result.crs).plot(
        ax=ax, facecolor="none", edgecolor="#222222", linewidth=0.6, zorder=99
    )
except Exception:
    pass

# Colorbar (let Matplotlib choose linear ticks)
sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap); sm._A = []
cbar = fig.colorbar(sm, ax=ax, fraction=0.030, pad=0.012)
cbar.formatter = FuncFormatter(lambda x, pos: f"{x:g}")
cbar.update_ticks()
cbar.minorticks_off()
cbar.ax.yaxis.set_minor_locator(NullLocator())
cbar.ax.tick_params(which="both", width=0.35, length=2.0, labelsize=6)
cbar.set_label("Avoided EAD (US$ millions)", fontsize=6.5, labelpad=4)

# Optional title (kept off for panel workflows)
# ax.set_title("Fig. 3(c) ... (linear scale)", fontsize=7, fontweight="bold", pad=6)

# --- Scale bar + North arrow --------------------------------------------------
Robyn_paper_2_defs.draw_scale_bar(ax, location=(0.88, 0.78), length_km=20, linewidth=0.6,
              label_offset=0.02, km_offset=0.01)
Robyn_paper_2_defs.draw_north_arrow(ax, location=(0.88, 0.86), size=0.05, fontsize=8, label_offset=0.02)


# Labels (Arial, 5 pt, with white halo)
NUM_FS, HALO_W = 5.0, 0.75
NUM_FP = fm.FontProperties(family="Arial", size=NUM_FS)
pts = result.geometry.representative_point()
try:
    mask = result.to_crs(3448).geometry.area >= 2e6
except Exception:
    mask = ~pts.is_empty

texts = []
for (x, y, uid, show) in zip(pts.x, pts.y, result["catchment_uid"].astype(str), mask):
    if not show:
        continue
    t = ax.text(
        x, y, uid, fontproperties=NUM_FP, ha="center", va="center", color="black",
        zorder=20, snap=True,
        path_effects=[pe.withStroke(linewidth=HALO_W, foreground="white",
                                    joinstyle="round", capstyle="round")]
    )
    texts.append(t)

try:
    from adjustText import adjust_text
    adjust_text(texts, ax=ax, only_move={'texts':'y'},
                expand_text=(1.08, 1.15), force_text=(0.06, 0.16))
except Exception:
    pass

plt.tight_layout()

# Save
fname = out_dir / "Fig3c_avoided_EAD_min_USD_millions_linear"
plt.savefig(fname.with_suffix(".png"), dpi=600, bbox_inches="tight", facecolor="white")
plt.savefig(fname.with_suffix(".pdf"),              bbox_inches="tight")
plt.show()
print("Saved:", fname.with_suffix(".png"))